# 03 — Results (plotting)

Run after `01_scenarios.ipynb`.

This notebook is the **plot** stage — it reads the CSV tables produced in
`results/tables/` and generates every figure in `results/figures/`. It does
not touch brightway or run any LCA, so plot iteration is fast.

Figures produced:

**Per scenario** (in `results/figures/{scenario}/`):
- `spatial_map.png` — 2×2 district-level LCIA maps (land use, water, N, P)
- `contribution.png` — supply-chain stage breakdown + top-15 districts
- `activity_contributions.png` — top-15 contributing activities per impact category
- `dls_spider.png` — DLS indicator coverage doughnut

**Cross-scenario** (in `results/figures/`):
- `maps_{land_use,water,n_eutro,p_eutro}.png` — one shared-scale figure per spatial category
- `maps_combined_regional.png` — combined regionalised impact, baseline-normalised
- `maps_combined_regional_diff.png` — baseline raw sum + per-scenario Δ vs baseline
- `scenario_comparison.png` — % change vs baseline by category × scenario
- `scenario_tradeoff.png` — wellbeing vs. biodiversity scatter

Runs after `02_monte_carlo` so the Monte Carlo figures can be built here too;
MC-dependent figures skip politely if the MC tables are absent.


In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

_prefix = sys.prefix
if os.name == "nt":  # Windows
    os.environ.setdefault("GDAL_DATA", os.path.join(_prefix, "Library", "share", "gdal"))
    os.environ.setdefault("PROJ_LIB",  os.path.join(_prefix, "Library", "share", "proj"))
else:
    os.environ.setdefault("GDAL_DATA", os.path.join(_prefix, "share", "gdal"))
    os.environ.setdefault("PROJ_LIB",  os.path.join(_prefix, "share", "proj"))

import pandas as pd
import numpy as np
import geopandas as gpd
gpd.options.io_engine = "fiona"
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.config import (
    RESULTS_TABLES_DIR, RESULTS_FIGURES_DIR,
    TZ_DISTRICTS_PATH,
)
from src.plotting import (
    plot_cross_scenario_maps,
    plot_contribution_breakdown, plot_contribution_analysis,
)

os.makedirs(RESULTS_FIGURES_DIR, exist_ok=True)

# Load Tanzania districts shapefile (used for joining scores to geometry)
tz_districts_gdf = gpd.read_file(TZ_DISTRICTS_PATH)
print(f"TZ districts: {len(tz_districts_gdf)} features, CRS={tz_districts_gdf.crs}")

# Load combined results tables
df          = pd.read_csv(os.path.join(RESULTS_TABLES_DIR, "all_scenarios.csv"))
dls_df      = pd.read_csv(os.path.join(RESULTS_TABLES_DIR, "dls_coverage_all_scenarios.csv"))
stage_df    = pd.read_csv(os.path.join(RESULTS_TABLES_DIR, "stage_scores_all_scenarios.csv"))
district_df = pd.read_csv(os.path.join(RESULTS_TABLES_DIR, "district_scores_all_scenarios.csv"))
display_df  = pd.read_csv(os.path.join(RESULTS_TABLES_DIR, "scenario_display.csv"))

# ── Water district outlier exclusion ─────────────────────────────────────────
# Chemba (TZ0107, Dodoma) sits in a PCR-GLOBWB basin with an extreme CF
# (~5 orders of magnitude above neighbouring districts), causing a
# +6,600,000% water impact in the drought scenario.  Excluded from all
# water district scores here pending a production-weighted CF averaging fix
# in 00_setup (TODO: replace area-weighted with production-weighted average).
_WATER_EXCLUDE = {"TZ0107"}
district_df = district_df[
    ~((district_df["district_id"].isin(_WATER_EXCLUDE)) &
      (district_df["category"] == "Water consumption"))
].copy()

scenario_display = dict(zip(display_df["scenario"], display_df["display"]))
scenarios        = list(scenario_display.keys())

print(f"Scenarios: {scenarios}")
print(f"Scores rows: {len(df)} | DLS rows: {len(dls_df)} | "
      f"Stage rows: {len(stage_df)} | District rows: {len(district_df)}")
df.head()

TZ districts: 170 features, CRS=EPSG:4326

Scenarios: ['baseline', 'high_yield', 'extensification', 'organic_expansion', 'manufacturing_expansion', 'irrigation']

Scores rows: 48 | DLS rows: 30 | Stage rows: 144 | District rows: 1554

,scenario,category,score,unit,approach
0,baseline,Land use occupation,1.803820e-05,PDF*m2*yr,OneSpatialScaleLCA + ecoregion CFs -> districts
1,baseline,Water consumption,1.874768e-10,PDF*yr,OneSpatialScaleLCA + PCR-GLOBWB basin CFs -> d...
2,baseline,FW eutrophication N,7.530358e-10,PDF*yr,OneSpatialScaleLCA + 0.5 deg raster CFs -> dis...
3,baseline,FW eutrophication P,1.119193e-09,PDF*yr,OneSpatialScaleLCA + 0.5 deg raster CFs -> dis...
4,baseline,Climate change rcp26,2.539345e-10,PDF*yr,bc.LCA + global NaturalEarth CFs


## SI contribution figures (baseline)

The per-scenario `spatial_map` / `dls_spider` figures and the non-baseline loops were
retired in the 2026-09 cleanup (superseded by the combined manuscript figures below).
What remains feeds the SI: `baseline/contribution.png` (stage breakdown) and
`baseline/activity_contributions.png` (Supplementary activity contribution figure).


In [2]:
# Spatial-category metadata: cat_key -> (cat_label_in_scores_csv, panel_title, unit, cmap)
SPATIAL_PANEL_META = {
    "land_use": ("Land use occupation",
                 "A.  Land Use Occupation\nLC-IMPACT, Regionalized Districts (ecoregion CFs)",
                 "District LCIA impact (PDF·yr)",
                 "YlOrRd"),
    "water":    ("Water consumption",
                 "B.  Water Consumption\nLC-IMPACT, Regionalized Districts (PCR-GLOBWB basin CFs)",
                 "District LCIA impact (PDF·yr)",
                 "Blues"),
    "n_eutro":  ("FW eutrophication N",
                 "C.  Freshwater Eutrophication — N\nLC-IMPACT, Regionalized Districts (diffuse CFs)",
                 "District LCIA impact (PDF·yr)",
                 "YlGnBu"),
    "p_eutro":  ("FW eutrophication P",
                 "D.  Freshwater Eutrophication — P\nLC-IMPACT, Regionalized Districts (diffuse CFs)",
                 "District LCIA impact (PDF·yr)",
                 "Greens"),
}

# Activity-contribution category labels (used for activity_contributions.png)
CA_CATS = [
    ("land_use", "Land use (regionalized)"),
    ("water",    "Water consumption"),
    ("n_eutro",  "FW eutrophication N"),
    ("p_eutro",  "FW eutrophication P"),
    ("climate",  "Climate change (rcp26)"),
    ("ecotox",   "FW ecotoxicity (USEtox)"),
]

# Contribution-breakdown panel categories: must match the labels in scores.csv
CB_CATS = [
    ("Land use",     "Land use occupation"),
    ("Water use",    "Water consumption"),
    ("Eutro. N",     "FW eutrophication N"),
    ("Eutro. P",     "FW eutrophication P"),
    ("Climate chg.", "Climate change rcp26"),
    ("FW ecotox.",   "FW ecotoxicity"),
]


def _district_gdf_for(scenario_name: str, cat_key: str):
    """District scores for one scenario and spatial category, merged with
    the district shapefile. Reads the combined long table (the per-scenario
    CSVs were retired in the 2026-09 cleanup); water rows already carry the
    Chemba exclusion applied to district_df in the setup cell."""
    cat_label = SPATIAL_PANEL_META[cat_key][0]
    df_d = district_df[(district_df["scenario"] == scenario_name)
                       & (district_df["category"] == cat_label)]
    df_d = df_d.rename(columns={"district_id": "ADM2_PCODE"})[
        ["ADM2_PCODE", "score"]]
    merged = tz_districts_gdf.merge(df_d, on="ADM2_PCODE", how="left")
    return merged, "score"


# Compute BASELINE category totals once. Panel B of the contribution figure
# uses these as the cross-scenario normaliser so bar heights are comparable
# across scenarios (e.g. mfg expansion shows much taller water-related bars
# in textile districts because water grew ~64×, even though within-scenario
# fractional shares are unchanged).
BASELINE_NAME = "baseline"
_baseline_score_df = df[df["scenario"] == BASELINE_NAME]
baseline_cat_totals = {}
for cb_label, score_label in CB_CATS:
    sub = _baseline_score_df[_baseline_score_df["category"] == score_label]
    if len(sub):
        baseline_cat_totals[cb_label] = float(sub["score"].iloc[0])

print("Baseline category totals (used as cross-scenario normaliser for panel B):")
for k, v in baseline_cat_totals.items():
    print(f"  {k:<14s} {v:.4e}")


for scen in ["baseline"]:
    print(f"\n=== {scenario_display[scen]} ({scen}) ===")
    fig_dir = os.path.join(RESULTS_FIGURES_DIR, scen)
    os.makedirs(fig_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # 2) contribution.png — stage + top-15 districts
    # ------------------------------------------------------------------
    scen_stage_df = stage_df[stage_df["scenario"] == scen]
    scen_dist_df  = district_df[district_df["scenario"] == scen]
    scen_score_df = df[df["scenario"] == scen]

    stage_scores_per_cat = {}
    district_scores_per_cat = {}
    cat_totals = {}
    cb_meta_filtered = []
    for cb_label, score_label in CB_CATS:
        sub_st = scen_stage_df[scen_stage_df["category"] == score_label]
        stage_scores_per_cat[cb_label] = dict(zip(sub_st["stage"], sub_st["score"]))

        sub_d = scen_dist_df[scen_dist_df["category"] == score_label]
        district_scores_per_cat[cb_label] = dict(zip(sub_d["district_id"], sub_d["score"]))

        # Total: from scores.csv if present, otherwise sum of stage scores
        sub_sc = scen_score_df[scen_score_df["category"] == score_label]
        if len(sub_sc):
            cat_totals[cb_label] = float(sub_sc["score"].iloc[0])
        else:
            cat_totals[cb_label] = sum(stage_scores_per_cat[cb_label].values())

        cb_meta_filtered.append((cb_label, None))

    plot_contribution_breakdown(
        stage_scores_per_cat=stage_scores_per_cat,
        district_scores_per_cat=district_scores_per_cat,
        cat_totals=cat_totals,
        cat_meta=cb_meta_filtered,
        tz_districts_gdf=tz_districts_gdf,
        out_path=os.path.join(fig_dir, "contribution.png"),
        top_district_n=15,
        baseline_cat_totals=baseline_cat_totals,
    )

    # ------------------------------------------------------------------
    # 3) activity_contributions.png — top-15 activities per category
    # ------------------------------------------------------------------
    ca_results = []
    for cat_key, cat_label in CA_CATS:
        ca_path = os.path.join(RESULTS_TABLES_DIR, scen, f"contribution_{cat_key}.csv")
        if os.path.exists(ca_path):
            ca_df = pd.read_csv(ca_path)
            ca_results.append((cat_label, ca_df))
    if ca_results:
        plot_contribution_analysis(
            ca_results,
            os.path.join(RESULTS_FIGURES_DIR, "fig_S5.png"),
            top_n=15,
        )


Baseline category totals (used as cross-scenario normaliser for panel B):

  Land use       1.8038e-05

  Water use      1.8748e-10

  Eutro. N       7.5304e-10

  Eutro. P       1.1192e-09

  Climate chg.   2.5393e-10

  FW ecotox.     5.2067e-06


=== Baseline (baseline) ===

Contribution figure saved -> X:\Eli\projects\tz_cotton\python\results\figures\baseline\contribution.png

Contribution analysis figure saved -> X:\Eli\projects\tz_cotton\python\results\figures\fig_S5.png

## Cross-scenario figures

Single-figure summaries that compare all scenarios:

- `maps_*.png` — one figure per spatial impact category, all scenarios on a shared colour scale.
- `maps_combined_regional.png` — combined regionalised impact per district, baseline-normalised then summed. Highlights *relative* growth from baseline (water-driven hotspots in mfg expansion stand out).
- `maps_combined_regional_diff.png` — baseline panel shows raw sum of regional impacts per district (PDF·yr), and each other panel shows scenario − baseline (red = added damage, blue = avoided damage). Direct, unit-consistent view of *where* each scenario adds or removes biodiversity damage.
- `scenario_comparison.png` — % change vs baseline by impact category × scenario (bar chart).
- `scenario_tradeoff.png` — wellbeing vs. biodiversity scatter. Y-axis is the % change in summed biodiversity damage; X-axis is the mean of per-indicator percent change in DLS coverage. Bottom-right quadrant = win-win.

In [3]:
# ── Cross-scenario maps (one figure per impact category, shared scale) ──────
_CROSS_CATS = [
    ("land_use", "Land Use Occupation",   "District LCIA impact (PDF·yr)", "YlOrRd"),
    ("water",    "Water Consumption",     "District LCIA impact (PDF·yr)",    "Blues"),
    ("n_eutro",  "FW Eutrophication — N", "District LCIA impact (PDF·yr)",    "YlGnBu"),
    ("p_eutro",  "FW Eutrophication — P", "District LCIA impact (PDF·yr)",    "Greens"),
]

for cat_key, cat_name, unit_label, cmap_name in _CROSS_CATS:
    scenario_gdfs = []
    for scen in scenarios:
        gdf_m, sc_col = _district_gdf_for(scen, cat_key)
        scenario_gdfs.append((scenario_display[scen], gdf_m, sc_col))
    out = os.path.join(RESULTS_FIGURES_DIR, f"maps_{cat_key}.png")
    plot_cross_scenario_maps(
        cat_name, unit_label, scenario_gdfs,
        tz_districts_gdf, out, cmap_name=cmap_name)


# ── Cross-scenario % change vs baseline (bar chart) ─────────────────────────
plot_df = df[~df["category"].str.contains("Ratio|site-generic", regex=True)].copy()
pivot_abs = plot_df.pivot_table(index="category", columns="scenario",
                                values="score", aggfunc="first")
baseline_vals = pivot_abs["baseline"]
pivot_pct = ((pivot_abs.div(baseline_vals, axis=0)) - 1) * 100.0

non_baseline = [c for c in pivot_pct.columns if c != "baseline"]
categories = pivot_pct.index.tolist()
n_cats  = len(categories)
n_scens = len(non_baseline)
x       = np.arange(n_cats)
width   = 0.7 / n_scens

scenario_colors = {
    "high_yield":               "#1F77B4",
    "organic_expansion":        "#2CA02C",
    "manufacturing_expansion":  "#9467BD",
}

fig, ax = plt.subplots(figsize=(14, 5), dpi=120)
for i, sc in enumerate(non_baseline):
    vals = [pivot_pct.loc[cat, sc] if sc in pivot_pct.columns else 0.0 for cat in categories]
    ax.bar(x + (i - n_scens / 2 + 0.5) * width, vals, width * 0.9,
           label=scenario_display.get(sc, sc), color=scenario_colors.get(sc, f"C{i}"))

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("% change from baseline")
ax.set_title("Tanzania Cotton LCA — Impact Change Relative to Baseline")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.0f}%"))
ax.legend(title="Scenario", fontsize=8)
ax.grid(axis="y", linestyle="--", linewidth=0.3, alpha=0.7)
plt.tight_layout()

fig_path = os.path.join(RESULTS_FIGURES_DIR, "scenario_comparison.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved -> {fig_path}")
print("\n% change from baseline:")
print(pivot_pct[non_baseline].to_string(float_format=lambda v: f"{v:+.1f}%"))

X:\Eli\projects\tz_cotton\python\src\plotting.py:198: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.91, 1.0])


Cross-scenario map saved -> X:\Eli\projects\tz_cotton\python\results\figures\maps_land_use.png

X:\Eli\projects\tz_cotton\python\src\plotting.py:198: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.91, 1.0])


Cross-scenario map saved -> X:\Eli\projects\tz_cotton\python\results\figures\maps_water.png

X:\Eli\projects\tz_cotton\python\src\plotting.py:198: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.91, 1.0])


...

X:\Eli\projects\tz_cotton\python\src\plotting.py:198: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.91, 1.0])


Cross-scenario map saved -> X:\Eli\projects\tz_cotton\python\results\figures\maps_p_eutro.png

Saved -> X:\Eli\projects\tz_cotton\python\results\figures\scenario_comparison.png


% change from baseline:

scenario              extensification  high_yield   irrigation  manufacturing_expansion  organic_expansion
category                                                                                                  
Climate change rcp26          +286.6%     +286.4%      +123.7%                    +2.3%              +0.1%
FW ecotoxicity                +281.7%     +281.0%      +120.7%                    +5.7%              -0.5%
FW eutrophication N           +289.9%     +289.9%        -0.0%                    +0.0%             +54.9%
FW eutrophication P           +289.9%     +289.9%        -0.0%                    +0.0%            +100.0%
Land use occupation           +289.9%       +0.0%        +0.0%                    +0.1%              +0.0%
Water consumption               +0.0%       +0.0% +17006197.2%                  +200.0%              +0.0%

## Manuscript and SI figures

All main-text figures (1–7) and the standalone SI figures, integrated into
`src/plotting.py` from the former standalone scripts (originals in `attic/`).
Each function is self-contained and reads only `results/tables/`.


In [4]:
from src import plotting as P

# ── Main-text figures ────────────────────────────────────────────
P.make_fig1_system_diagram()        # Fig 1  system diagram
P.make_fig2_scenario_propagation()  # Fig 2  scenario propagation
P.make_fig3_disaggregation()        # Fig 3  impact disaggregation
P.make_fig4_dotplot_uncertainty()   # Fig 4  dot plot (needs MC tables)
P.make_fig5_delta_maps()            # Fig 5  scenario delta maps
P.make_fig6_dls_spider()            # Fig 6  DLS spider
P.make_fig7_tradeoff()              # Fig 7  trade-off

# ── SI figures ───────────────────────────────────────────────────
P.make_si_sankey()                  # material flows (plotly + kaleido)
P.make_si_chemba_map()              # Chemba outlier district
P.make_si_mc_plots("baseline")      # MC boxplots (needs MC tables)

# ── Exploratory variants (not in the manuscript) ─────────────────
MAKE_EXPLORATORY = True
if MAKE_EXPLORATORY:
    P.make_fig2b_scenario_bubbles()
    P.make_fig3c_hierarchy()


Saved -> X:\Eli\projects\tz_cotton\python\results\figures\fig_1.png

Saved -> X:\Eli\projects\tz_cotton\python\results\figures\fig_2.png

Saved -> X:\Eli\projects\tz_cotton\python\results\figures\fig_3.png


national combined total: 1.8040e-05 PDF·yr

  b) Land use            1.8038e-05  (99.9886%)  n=93

...

Irrigation                        +0.0          -0.0          -0.0        +123.7        +120.7      +19723.7        +154.2          +0.0        +122.4        +125.7        +125.7

Saved -> X:\Eli\projects\tz_cotton\python\results\figures\fig_3_c.png


national combined total: 1.8040e-05 PDF·yr

  terrestrial (land use)  1.8038e-05  ( 99.9886%)

  freshwater (W+N+P)      2.0597e-09  (  0.0114%)